In [ ]:
from modeling_module.utils.date_util import DateUtil
import polars as pl
def augment_data(demand_sales):
    rng = (
        demand_sales
            .sort(['part_no', 'yyyyww'])
            .group_by('part_no')
            .agg([
                pl.col('yyyyww_right').min().alias('demand_start_pos'),
                pl.col('yyyyww_right').max().alias('demand_end_pos'),
                pl.col('yyyyww').min().alias('sales_start_pos')
            ])
            .with_columns(pl.lit('202627').alias('demand_end_pos'))
            .with_columns(
                [
                    pl.col('sales_start_pos')
                      .map_elements(DateUtil.yyyyww_to_date, return_dtype=pl.Date)
                      .alias('start_date'),
                    pl.col('demand_end_pos')
                      .map_elements(DateUtil.yyyyww_to_date, return_dtype=pl.Date)
                      .alias('end_date')
                ]
            )
            .with_columns(
                pl.date_ranges(start = 'start_date', end = 'end_date', interval='1w').alias('date')
            )
            .explode('date')
            .with_columns(pl.col('date').map_elements(DateUtil.date_to_yyyyww, return_dtype=pl.Int64).alias('yyyyww'))
            .select(['part_no', 'demand_start_pos', 'sales_start_pos', 'yyyyww'])
    )
    rng.select(['part_no', 'demand_start_pos', 'sales_start_pos', 'yyyyww'])
    se_pos = rng.join(demand_sales, on = ['part_no', 'yyyyww'], how = 'left').with_columns(pl.col('order_qty').fill_null(0))
    df_expanded = (
        se_pos.select(['part_no', 'yyyyww', 'order_qty', 'demand_qty', 'demand_start_pos', 'sales_start_pos'])
              .with_columns(pl.when(pl.col('yyyyww') < pl.col('demand_start_pos')).then(pl.lit(0)).otherwise(pl.lit(1)).alias('before_demand'))
              .with_columns(
                [
                    pl.when(pl.col('before_demand') == 0)
                      .then(None)
                      .otherwise(pl.col('demand_qty').fill_null(0))
                      .alias('demand_qty'),
                    pl.col('order_qty').fill_null(0)
                ]
              )
              .rename({'yyyyww': 'yyyyww_re'})
    )

    result_add = (
        df_expanded
            .filter((pl.col('sales_start_pos') < 201801))
            .with_columns([pl.col('yyyyww_re').shift(364).over('part_no').alias('demand_ago')])
            .join(df_expanded.select(['part_no', 'yyyyww_re', 'demand_qty', 'order_qty', 'before_demand']),
                  left_on = ['part_no', 'demand_ago'],
                  right_on = ['part_no', 'yyyyww_re'],
                  how = 'left'
                  )
            .with_columns(pl.when(pl.col('before_demand_right') == 0).then(pl.lit(-1)).otherwise(pl.col('demand_qty_right')).alias('demand_qty'))
            .drop_nulls(pl.col('demand_qty'))
            .with_columns((pl.col('part_no') + '_add').alias('part_no'))
            .select(['part_no', 'yyyyww_re', 'demand_qty', 'order_qty_right'])
            .rename({'order_qty_right': 'order_qty'})
            .with_columns([
                pl.col('yyyyww_re').min().over('part_no').alias('sales_start_pos'),
                pl.when(pl.col('demand_qty') >= 0)
                  .then(pl.col('yyyyww_re'))
                  .otherwise(None)
                  .min()
                  .over('part_no')
                  .alias('demand_start_pos'),
                pl.col('yyyyww_re').max().over('part_no').alias('demand_end_pos')
           ])
    )

    result_add11 = (
        df_expanded
            .filter((pl.col('sales_start_pos') < 202001) & (pl.col('sales_start_pos') >= 201802))
            .with_columns([pl.col('yyyyww_re').shift(260).over('part_no').alias('demand_ago')])
            .join(df_expanded.select(['part_no', 'yyyyww_re', 'demand_qty', 'order_qty', 'before_demand']),
                  left_on = ['part_no', 'demand_ago'],
                  right_on = ['part_no', 'yyyyww_re'],
                  how = 'left'
                  )
            .with_columns(pl.when(pl.col('before_demand_right') == 0).then(pl.lit(-1)).otherwise(pl.col('demand_qty_right')).alias('demand_qty'))
            .drop_nulls(pl.col('demand_qty'))
            .with_columns((pl.col('part_no') + '_add').alias('part_no'))
            .select(['part_no', 'yyyyww_re', 'demand_qty', 'order_qty_right'])
            .rename({'order_qty_right': 'order_qty'})
            .with_columns([
                pl.col('yyyyww_re').min().over('part_no').alias('sales_start_pos'),
                pl.when(pl.col('demand_qty') >= 0)
                  .then(pl.col('yyyyww_re'))
                  .otherwise(None)
                  .min()
                  .over('part_no')
                  .alias('demand_start_pos'),
                pl.col('yyyyww_re').max().over('part_no').alias('demand_end_pos')
           ])
    )

    result_add22 = (
        df_expanded
            .filter((pl.col('sales_start_pos') < 202101) & (pl.col('sales_start_pos') >= 202002))
            .with_columns([pl.col('yyyyww_re').shift(220).over('part_no').alias('demand_ago')])
            .join(df_expanded.select(['part_no', 'yyyyww_re', 'demand_qty', 'order_qty', 'before_demand']),
                  left_on = ['part_no', 'demand_ago'],
                  right_on = ['part_no', 'yyyyww_re'],
                  how = 'left'
                  )
            .with_columns(pl.when(pl.col('before_demand_right') == 0).then(pl.lit(-1)).otherwise(pl.col('demand_qty_right')).alias('demand_qty'))
            .drop_nulls(pl.col('demand_qty'))
            .with_columns((pl.col('part_no') + '_add').alias('part_no'))
            .select(['part_no', 'yyyyww_re', 'demand_qty', 'order_qty_right'])
            .rename({'order_qty_right': 'order_qty'})
            .with_columns([
                pl.col('yyyyww_re').min().over('part_no').alias('sales_start_pos'),
                pl.when(pl.col('demand_qty') >= 0)
                  .then(pl.col('yyyyww_re'))
                  .otherwise(None)
                  .min()
                  .over('part_no')
                  .alias('demand_start_pos'),
                pl.col('yyyyww_re').max().over('part_no').alias('demand_end_pos')
           ])
    )

    result_add33 = (
        df_expanded
            .filter((pl.col('sales_start_pos') < 202201) & (pl.col('sales_start_pos') >= 202102))
            .with_columns([pl.col('yyyyww_re').shift(180).over('part_no').alias('demand_ago')])
            .join(df_expanded.select(['part_no', 'yyyyww_re', 'demand_qty', 'order_qty', 'before_demand']),
                  left_on = ['part_no', 'demand_ago'],
                  right_on = ['part_no', 'yyyyww_re'],
                  how = 'left'
                  )
            .with_columns(pl.when(pl.col('before_demand_right') == 0).then(pl.lit(-1)).otherwise(pl.col('demand_qty_right')).alias('demand_qty'))
            .drop_nulls(pl.col('demand_qty'))
            .with_columns((pl.col('part_no') + '_add').alias('part_no'))
            .select(['part_no', 'yyyyww_re', 'demand_qty', 'order_qty_right'])
            .rename({'order_qty_right': 'order_qty'})
            .with_columns([
                pl.col('yyyyww_re').min().over('part_no').alias('sales_start_pos'),
                pl.when(pl.col('demand_qty') >= 0)
                  .then(pl.col('yyyyww_re'))
                  .otherwise(None)
                  .min()
                  .over('part_no')
                  .alias('demand_start_pos'),
                pl.col('yyyyww_re').max().over('part_no').alias('demand_end_pos')
           ])
    )

    result_add2 = (
        df_expanded
            .filter((pl.col('sales_start_pos') >= 202301) & (pl.col('sales_start_pos') < 202401))
            .with_columns([pl.col('yyyyww_re').shift(78).over('part_no').alias('demand_ago')])
            .join(df_expanded.select(['part_no', 'yyyyww_re', 'demand_qty', 'order_qty', 'before_demand']),
                  left_on = ['part_no', 'demand_ago'],
                  right_on = ['part_no', 'yyyyww_re'],
                  how = 'left'
                  )
            .with_columns(pl.when(pl.col('before_demand_right') == 0).then(pl.lit(-1)).otherwise(pl.col('demand_qty_right')).alias('demand_qty'))
            .drop_nulls(pl.col('demand_qty'))
            .with_columns((pl.col('part_no') + '_add').alias('part_no'))
            .select(['part_no', 'yyyyww_re', 'demand_qty', 'order_qty_right'])
            .rename({'order_qty_right': 'order_qty'})
            .with_columns([
                pl.col('yyyyww_re').min().over('part_no').alias('sales_start_pos'),
                pl.when(pl.col('demand_qty') >= 0)
                  .then(pl.col('yyyyww_re'))
                  .otherwise(None)
                  .min()
                  .over('part_no')
                  .alias('demand_start_pos'),
                pl.col('yyyyww_re').max().over('part_no').alias('demand_end_pos')
           ])
    )

    result_add3 = (
        df_expanded
            .filter(pl.col('sales_start_pos') >= 202401)
            .with_columns([pl.col('yyyyww_re').shift(78).over('part_no').alias('demand_ago')])
            .join(df_expanded.select(['part_no', 'yyyyww_re', 'demand_qty', 'order_qty', 'before_demand']),
                  left_on = ['part_no', 'demand_ago'],
                  right_on = ['part_no', 'yyyyww_re'],
                  how = 'left'
                  )
            .with_columns(pl.when(pl.col('before_demand_right') == 0).then(pl.lit(-1)).otherwise(pl.col('demand_qty_right')).alias('demand_qty'))
            .drop_nulls(pl.col('demand_qty'))
            .with_columns((pl.col('part_no') + '_add').alias('part_no'))
            .select(['part_no', 'yyyyww_re', 'demand_qty', 'order_qty_right'])
            .rename({'order_qty_right': 'order_qty'})
            .with_columns([
                pl.col('yyyyww_re').min().over('part_no').alias('sales_start_pos'),
                pl.when(pl.col('demand_qty') >= 0)
                  .then(pl.col('yyyyww_re'))
                  .otherwise(None)
                  .min()
                  .over('part_no')
                  .alias('demand_start_pos'),
                pl.col('yyyyww_re').max().over('part_no').alias('demand_end_pos')
           ])
    )

    result_long_ad = (
        pl.concat([df_expanded, result_add33, result_add3, result_add, result_add22, result_add2, result_add11], how = 'diagonal')
          .sort('part_no', 'yyyyww_re')
          .with_columns(pl.col('yyyyww_re').len().over('part_no').alias('max'))
          .with_columns(pl.when(pl.col('demand_qty') == -1).then(None).otherwise(pl.col('demand_qty')).alias('demand_qty'))
    )

    return result_long_ad


In [ ]:
def make_result_exog(result_long_ad, data):
    result_fix = (
        result_long_ad
            .rename({'yyyyww_re': 'yyyyww'})
            .sort(['part_no', 'yyyyww'])
            .with_columns(pl.col('part_no').str.replace('_add', '').alias('part_no_base'))
            .with_columns([
                pl.col('demand_qty').cum_sum().over('part_no').alias('demand_cumsum'),
                pl.col('order_qty').cum_sum().over('part_no').alias('order_cumsum'),
            ])
            .with_columns([
                (pl.col('demand_cumsum').max()).over('part_no').alias('demand_cumsum_max'),
                (pl.col('demand_cumsum').min()).over('part_no').alias('demand_cumsum_min'),
                (pl.col('order_cumsum').max()).over('part_no').alias('order_cumsum_max'),
                (pl.col('order_cumsum').min()).over('part_no').alias('order_cumsum_min'),
                (pl.col('demand_qty').max()).over('part_no').alias('demand_qty_max'),
                (pl.col('demand_qty').min()).over('part_no').alias('demand_qty_min'),
                (pl.col('order_qty').max()).over('part_no').alias('order_qty_max'),
                (pl.col('order_qty').min()).over('part_no').alias('order_qty_max'),
            ])
            .filter(
                (pl.col('demand_cumsum_max') != pl.col('demand_cumsum_min'))
                & (pl.col('order_cumsum_max') != pl.col('order_cumsum_min'))
                & (pl.col('demand_qty_max') != pl.col('demand_qty_min'))
                & (pl.col('order_qty_max') != pl.col('order_qty_min'))
            )
            .with_columns([
                ((pl.col('demand_cumsum') - pl.col('demand_cumsum_min')) / (pl.col('demand_cumsum_max') - pl.col('demand_cumsum_min'))).over('part_no').alias('demand_cumsum_mm'),
                ((pl.col('order_cumsum') - pl.col('order_cumsum_min')) / (pl.col('order_cumsum_max') - pl.col('order_cumsum_min'))).over('part_no').alias('order_cumsum_mm'),
                ((pl.col('demand_qty') - pl.col('demand_qty_min')) / (pl.col('demand_qty_max') - pl.col('demand_qty_min'))).over('part_no').alias('demand_qty_mm'),
                ((pl.col('order_qty') - pl.col('order_qty_min')) / (pl.col('order_qty_max') - pl.col('order_qty_min'))).over('part_no').alias('order_qty_mm'),
                pl.col('demand_qty').log1p().alias('demand_qty_log'),
                pl.col('order_qty').log1p().alias('order_qty_log'),
                pl.col('order_cumsum').log1p().alias('order_cumsum_log'),
                pl.col('demand_cumsum').log1p().alias('demand_cumsum_log')
            ])
            .sort(['part_no', 'yyyyww'])
    )

    result_exog = (
        result_fix.join(data, left_on = 'part_no_base', right_on = 'part_no', how = 'left')
                  .with_columns([
                    (pl.col('stock_keep_time') / 12 * 52).cast(pl.Int64).alias('replacement'),
                    (pl.col('stock_keep_time') / 12 * 52).cast(pl.Int64).alias('retention'),
                    (pl.col('wty_month') / 12 * 52).cast(pl.Int64).alias('warranty')
                  ])
                  .with_columns([
                    pl.struct(['sales_start_pos', 'replacement'])
                      .map_elements(lambda s: DateUtil.add_week(s['sales_start_pos'], s['replacement']), return_dtype = pl.Utf8)
                      .alias('replacement_end'),
                    pl.struct(['sales_start_pos', 'retention'])
                      .map_elements(lambda s: DateUtil.add_week(s['sales_start_pos'], s['retention']), return_dtype = pl.Utf8)
                      .alias('retention_end'),
                    pl.struct(['sales_start_pos', 'warranty'])
                      .map_elements(lambda s: DateUtil.add_week(s['sales_start_pos'], s['warranty']), return_dtype = pl.Utf8)
                      .alias('warranty_end')
                 ])
                 .sort(['part_no', 'yyyyww'])
                 .with_columns([
                    pl.col('demand_cumsum').shift(pl.col('warranty').first() + 1).over('part_no').alias('demand_cumsum_ago'),
                    pl.col('demand_cumsum_mm').shift(pl.col('warranty').first() + 1).over('part_no').alias('demand_cumsum_mm_ago'),
                    pl.col('demand_cumsum_log').shift(pl.col('warranty').first() + 1).over('part_no').alias('demand_cumsum_log_ago'),
                    pl.col('demand_cumsum_mm').shift(pl.col('replacement').first() + 1).over('part_no').alias('demand_replace_mm_ago'),
                    pl.col('demand_cumsum_log').shift(pl.col('replacement').first() + 1).over('part_no').alias('demand_replace_log_ago'),
                    pl.col('order_cumsum_log').shift(pl.col('warranty').first() + 1).over('part_no').alias('order_cumsum_log_ago'),
                    pl.col('order_cumsum').shift(pl.col('warranty').first() + 1).over('part_no').alias('order_cumsum_ago'),
                    pl.col('order_cumsum').shift(pl.col('replacement').first() + 1).over('part_no').alias('order_cumsum_replacement_ago'),
                ])
                .drop_nulls(['demand_cumsum'])
                .with_columns([
                    pl.col('warranty_end').map_elements(lambda s: DateUtil.add_week(s, -26), return_dtype = pl.Utf8).cast(pl.Int64).alias('wty_ago27')
                ])
                .with_columns(pl.when(pl.col('yyyyww') > pl.col('warranty_end').cast(pl.Int64))
                              .then((pl.col('order_cumsum') - pl.col('order_cumsum_ago')).log1p())
                              .otherwise(pl.col('order_cumsum').log1p())
                              .alias('in_wty_log'))
                .with_columns(pl.when(pl.col('yyyyww') > pl.col('warranty_end').cast(pl.Int64))
                              .then((pl.col('order_cumsum') - pl.col('order_cumsum_ago')))
                              .otherwise(pl.col('order_cumsum'))
                              .alias('in_wty'))
                .sort(['part_no', 'yyyyww'])
                .with_columns(pl.when(pl.col('yyyyww') >= pl.col('replacement_end').cast(pl.Int64))
                              .then(pl.col('order_cumsum') - pl.col('order_cumsum_replace_ago'))
                              .otherwise(pl.col('order_cumsum'))
                              .alias('active'))
                .with_columns(pl.when((pl.col('yyyyww') > pl.col('warranty_end').cast(pl.Int64)) & (pl.col('yyyyww') <= pl.col('replacement_end').cast(pl.Int64)))
                              .then(pl.col('order_cumsum_ago').log1p())
                              .otherwise(0)
                              .alias('out_wty_log'))
                .with_columns(pl.when((pl.col('yyyyww') > pl.col('warranty_end').cast(pl.Int64)) & (pl.col('yyyyww') <= pl.col('replacement_end').cast(pl.Int64)))
                              .then(pl.col('order_cumsum_ago'))
                              .otherwise(0)
                              .alias('out_wty'))
    )

    return result_exog

In [ ]:
from modeling_module.utils.date_util import DateUtil
import polars as pl


TARGET_END_YYYYWW = 202627

# 경계 누락 보정 기준
# - 201801  -> 260 bucket 포함
# - 202001  -> 220 bucket 포함
# - 202101  -> 180 bucket 포함
# - 202201 이후 -> 78 bucket 으로 통합
AUGMENT_RULES = [
    {"start": None,   "end": 201801, "shift": 364},
    {"start": 201801, "end": 202001, "shift": 260},
    {"start": 202001, "end": 202101, "shift": 220},
    {"start": 202101, "end": 202201, "shift": 180},
    {"start": 202201, "end": None,   "shift": 78},
]


def _build_sales_start_filter(start: int | None, end: int | None) -> pl.Expr:
    expr = pl.lit(True)

    if start is not None:
        expr = expr & (pl.col("sales_start_pos") >= pl.lit(start))

    if end is not None:
        expr = expr & (pl.col("sales_start_pos") < pl.lit(end))

    return expr


def _build_expanded_base(demand_sales: pl.DataFrame) -> pl.DataFrame:
    """
    part_no별로 sales_start_pos ~ TARGET_END_YYYYWW 까지 주차를 확장하고,
    원본 demand/order 값을 붙여 base frame 생성
    """
    rng = (
        demand_sales
        .group_by("part_no")
        .agg([
            pl.col("yyyyww_right").min().alias("demand_start_pos"),
            pl.col("yyyyww").min().alias("sales_start_pos"),
        ])
        .with_columns([
            pl.lit(TARGET_END_YYYYWW).alias("demand_end_pos"),
            pl.col("sales_start_pos")
              .map_elements(DateUtil.yyyyww_to_date, return_dtype=pl.Date)
              .alias("start_date"),
            pl.lit(TARGET_END_YYYYWW)
              .map_elements(DateUtil.yyyyww_to_date, return_dtype=pl.Date)
              .alias("end_date"),
        ])
        .with_columns(
            pl.date_ranges(
                start=pl.col("start_date"),
                end=pl.col("end_date"),
                interval="1w"
            ).alias("date")
        )
        .explode("date")
        .with_columns(
            pl.col("date")
              .map_elements(DateUtil.date_to_yyyyww, return_dtype=pl.Int64)
              .alias("yyyyww")
        )
        .select([
            "part_no",
            "demand_start_pos",
            "demand_end_pos",
            "sales_start_pos",
            "yyyyww",
        ])
    )

    df_expanded = (
        rng
        .join(demand_sales, on=["part_no", "yyyyww"], how="left")
        .with_columns([
            pl.col("order_qty").fill_null(0).alias("order_qty"),
        ])
        .select([
            "part_no",
            "yyyyww",
            "order_qty",
            "demand_qty",
            "demand_start_pos",
            "demand_end_pos",
            "sales_start_pos",
        ])
        .with_columns([
            pl.when(pl.col("yyyyww") < pl.col("demand_start_pos"))
              .then(pl.lit(0))
              .otherwise(pl.lit(1))
              .alias("before_demand"),
        ])
        .with_columns([
            pl.when(pl.col("before_demand") == 0)
              .then(None)
              .otherwise(pl.col("demand_qty").fill_null(0))
              .alias("demand_qty"),
        ])
        .rename({"yyyyww": "yyyyww_re"})
        .select([
            "part_no",
            "yyyyww_re",
            "order_qty",
            "demand_qty",
            "demand_start_pos",
            "demand_end_pos",
            "sales_start_pos",
            "before_demand",
        ])
    )

    return df_expanded


def _build_augmented_block(
    df_expanded: pl.DataFrame,
    start: int | None,
    end: int | None,
    shift_weeks: int,
) -> pl.DataFrame:
    """
    특정 sales_start_pos 구간에 대해 shift_weeks 만큼 과거 row를 참조해
    synthetic(_add) part를 생성
    """
    filter_expr = _build_sales_start_filter(start, end)

    source_df = df_expanded.select([
        "part_no",
        pl.col("yyyyww_re").alias("src_yyyyww_re"),
        pl.col("demand_qty").alias("src_demand_qty"),
        pl.col("order_qty").alias("src_order_qty"),
        pl.col("before_demand").alias("src_before_demand"),
    ])

    augmented = (
        df_expanded
        .filter(filter_expr)
        .with_columns([
            pl.col("yyyyww_re")
              .shift(shift_weeks)
              .over("part_no")
              .alias("demand_ago"),
        ])
        .join(
            source_df,
            left_on=["part_no", "demand_ago"],
            right_on=["part_no", "src_yyyyww_re"],
            how="left",
        )
        .with_columns([
            # 기존 코드의 -1 sentinel 로직을 제거하고,
            # keep_row를 별도로 둬서 before_demand row는 유지하되 demand_qty는 None 유지
            pl.when(pl.col("src_before_demand") == 0)
              .then(None)
              .otherwise(pl.col("src_demand_qty"))
              .alias("demand_qty"),
            pl.when(pl.col("src_before_demand") == 0)
              .then(pl.lit(True))
              .otherwise(pl.col("src_demand_qty").is_not_null())
              .alias("keep_row"),
        ])
        .filter(pl.col("keep_row"))
        .with_columns([
            pl.concat_str([pl.col("part_no"), pl.lit("_add")]).alias("part_no"),
            pl.col("src_order_qty").alias("order_qty"),
        ])
        .select([
            "part_no",
            "yyyyww_re",
            "order_qty",
            "demand_qty",
        ])
        .with_columns([
            pl.col("yyyyww_re").min().over("part_no").alias("sales_start_pos"),
            pl.when(pl.col("demand_qty").is_not_null())
              .then(pl.col("yyyyww_re"))
              .otherwise(None)
              .min()
              .over("part_no")
              .alias("demand_start_pos"),
            pl.col("yyyyww_re").max().over("part_no").alias("demand_end_pos"),
            pl.when(pl.col("demand_qty").is_null())
              .then(pl.lit(0))
              .otherwise(pl.lit(1))
              .alias("before_demand"),
        ])
        .select([
            "part_no",
            "yyyyww_re",
            "order_qty",
            "demand_qty",
            "demand_start_pos",
            "demand_end_pos",
            "sales_start_pos",
            "before_demand",
        ])
    )

    return augmented


def augment_data(demand_sales: pl.DataFrame) -> pl.DataFrame:
    """
    원본 demand_sales를 base 확장한 뒤,
    sales_start_pos 구간별 augmentation rule을 적용하여 synthetic part를 추가 생성
    """
    demand_sales = demand_sales.with_columns([
        pl.col("yyyyww").cast(pl.Int64),
        pl.col("yyyyww_right").cast(pl.Int64),
    ])

    df_expanded = _build_expanded_base(demand_sales)

    augmented_blocks = [
        _build_augmented_block(
            df_expanded=df_expanded,
            start=rule["start"],
            end=rule["end"],
            shift_weeks=rule["shift"],
        )
        for rule in AUGMENT_RULES
    ]

    result_long_ad = (
        pl.concat([df_expanded, *augmented_blocks], how="vertical_relaxed")
        .sort(["part_no", "yyyyww_re"])
        .with_columns(
            pl.col("yyyyww_re").len().over("part_no").alias("max")
        )
    )

    return result_long_ad